## Quality Checks Notebook — Data, API, Model Artifacts, and Dashboard

This notebook runs reproducible quality checks for the diabetes prediction project. It is intended to be a single, readable place to:

- Validate dataset integrity and schema
- Inspect model artifacts (`pipeline.pkl`, `meta.pkl`)
- Perform API health checks


### 1. Setup & Imports

In [53]:
import pandas as pd
import numpy as np
import requests
import sys
import time
import subprocess
from pathlib import Path
from sklearn.model_selection import train_test_split
import json
import joblib

## 2. Data Quality Checks

In [55]:
data_path = Path("../data/diabetes.csv")
cleaned_data_path = Path("../data/diabetes_cleaned.csv")

if data_path.exists():
    original_data = pd.read_csv(data_path)
    print(f"Original data loaded: {len(original_data)} rows, {len(original_data.columns)} columns")
else:
    print(f"Original data not found at {data_path}")
    
if cleaned_data_path.exists():
    cleaned_data = pd.read_csv(cleaned_data_path)
    print(f"Cleaned data loaded: {len(cleaned_data)} rows, {len(cleaned_data.columns)} columns")
else:
    print(f"Cleaned data not found at {cleaned_data_path}")

Original data loaded: 768 rows, 9 columns
Cleaned data loaded: 391 rows, 10 columns


Verify that all columns exist and values are within plausible ranges.


In [58]:
def check_schema_and_ranges(data):
    """Validate schema and check if values are within plausible ranges."""
    print("=" * 60)
    print("SCHEMA AND RANGE VALIDATION")
    print("=" * 60)
    
    if data is None:
        print("Cannot validate: data not loaded")
        return False
    
    # Expected columns and their plausible ranges (from preprocess.py)
    expected_schema = {
        'Pregnancies': {'dtype': 'int64', 'min': 0, 'max': 20},
        'Glucose': {'dtype': 'int64', 'min': 44, 'max': 200},
        'BloodPressure': {'dtype': 'int64', 'min': 24, 'max': 122},
        'SkinThickness': {'dtype': 'int64', 'min': 7, 'max': 99},
        'Insulin': {'dtype': 'int64', 'min': 14, 'max': 846},
        'BMI': {'dtype': 'float64', 'min': 18.2, 'max': 67.1},
        'DiabetesPedigreeFunction': {'dtype': 'float64', 'min': 0.078, 'max': 2.42},
        'Age': {'dtype': 'int64', 'min': 21, 'max': 81},
        'Outcome': {'dtype': 'int64', 'min': 0, 'max': 1}
    }
    
    all_valid = True
    
    # Check for missing columns
    missing_columns = set(expected_schema.keys()) - set(data.columns)
    if missing_columns:
        print(f"\n Missing columns: {missing_columns}")
        all_valid = False
    else:
        print("\n All expected columns present")
    
    print(f"\n Column-by-column validation:")
    print("-" * 60)
    
    # Check each column
    for col, specs in expected_schema.items():
        if col not in data.columns:
            continue
        
        # Check ranges
        col_min = data[col].min()
        col_max = data[col].max()
        
        range_valid = (col_min >= specs['min']) and (col_max <= specs['max'])
        
        status = "✓" if range_valid else "✗"
        print(f"{status} {col:25} | Range: [{col_min:>7.2f}, {col_max:>7.2f}] | Expected: [{specs['min']}, {specs['max']}]")
        
        if not range_valid:
            all_valid = False
    
    print("=" * 60)
    if all_valid:
        print("ALL SCHEMA AND RANGE CHECKS PASSED")
    else:
        print("SOME CHECKS FAILED")
    
    return all_valid

# Run the check
schema_check_passed = check_schema_and_ranges(cleaned_data)


SCHEMA AND RANGE VALIDATION

 All expected columns present

 Column-by-column validation:
------------------------------------------------------------
✓ Pregnancies               | Range: [   0.00,   17.00] | Expected: [0, 20]
✓ Glucose                   | Range: [  56.00,  198.00] | Expected: [44, 200]
✓ BloodPressure             | Range: [  30.00,  110.00] | Expected: [24, 122]
✓ SkinThickness             | Range: [   7.00,   63.00] | Expected: [7, 99]
✓ Insulin                   | Range: [  14.00,  846.00] | Expected: [14, 846]
✓ BMI                       | Range: [  18.20,   67.10] | Expected: [18.2, 67.1]
✓ DiabetesPedigreeFunction  | Range: [   0.09,    2.42] | Expected: [0.078, 2.42]
✓ Age                       | Range: [  21.00,   81.00] | Expected: [21, 81]
✓ Outcome                   | Range: [   0.00,    1.00] | Expected: [0, 1]
ALL SCHEMA AND RANGE CHECKS PASSED


In [59]:
def check_data_quality_metrics(data):
    """Check for missing values, duplicates, and other quality issues."""
    print("=" * 60)
    print("DATA QUALITY METRICS")
    print("=" * 60)
    
    if data is None:
        print("Cannot check quality: data not loaded")
        return False
    
    # Missing values
    print("\n1. Missing Values Check:")
    missing_counts = data.isnull().sum()
    total_missing = missing_counts.sum()
    
    if total_missing == 0:
        print(f"No missing values in cleaned dataset")
    else:
        print(f"Found {total_missing} missing values:")
        for col, count in missing_counts[missing_counts > 0].items():
            print(f"      - {col}: {count}")
    
    # Duplicate rows
    print("\n2. Duplicate Rows Check:")
    duplicate_count = data.duplicated().sum()
    if duplicate_count == 0:
        print(f"No duplicate rows")
    else:
        print(f"Found {duplicate_count} duplicate rows")
    
    # Outcome distribution
    print("\n3. Target Variable Distribution:")
    outcome_dist = data['Outcome'].value_counts().sort_index()
    print(f"   Non-diabetic (0): {outcome_dist.get(0, 0):>4} ({outcome_dist.get(0, 0)/len(data):>6.1%})")
    print(f"   Diabetic (1):     {outcome_dist.get(1, 0):>4} ({outcome_dist.get(1, 0)/len(data):>6.1%})")
    
    # Check for class imbalance
    imbalance_ratio = outcome_dist.max() / outcome_dist.min()
    print(f"\n   Class imbalance ratio: {imbalance_ratio:.2f}:1")
    
    print("=" * 60)
    quality_passed = (total_missing == 0) and (duplicate_count == 0)
    return quality_passed

# Run the check
quality_check_passed = check_data_quality_metrics(cleaned_data)

DATA QUALITY METRICS

1. Missing Values Check:
No missing values in cleaned dataset

2. Duplicate Rows Check:
No duplicate rows

3. Target Variable Distribution:
   Non-diabetic (0):  261 ( 66.8%)
   Diabetic (1):      130 ( 33.2%)

   Class imbalance ratio: 2.01:1


In [61]:
def compare_original_vs_cleaned(original, cleaned):
    """Compare original and cleaned datasets to verify preprocessing."""
    print("=" * 60)
    print("ORIGINAL vs CLEANED COMPARISON")
    print("=" * 60)
    
    if original is None or cleaned is None:
        print("Cannot compare: both datasets must be loaded")
        return
    
    print(f"\nOriginal dataset: {len(original):>4} rows, {len(original.columns)} columns")
    print(f"Cleaned dataset:  {len(cleaned):>4} rows, {len(cleaned.columns)} columns")
    
    rows_removed = len(original) - len(cleaned)
    removal_pct = (rows_removed / len(original)) * 100
    
    print(f"\nRows removed during cleaning: {rows_removed} ({removal_pct:.1f}%)")
        
    print("=" * 60)

compare_original_vs_cleaned(original_data, cleaned_data)

ORIGINAL vs CLEANED COMPARISON

Original dataset:  768 rows, 9 columns
Cleaned dataset:   391 rows, 10 columns

Rows removed during cleaning: 377 (49.1%)


## Model artifact checks (PKL files)

This section inspects the trained model artifacts stored under `models/`:

- `pipeline.pkl` — the preprocessing + model pipeline
- `meta.pkl` — metadata including selected model, CV scores, feature columns, and feature importance

In [64]:
def quality_check_artifacts(pipeline_path="../models/pipeline.pkl", meta_path="../models/meta.pkl"):
    """Check that trained model artifacts are valid and report selected model & parameters."""
    
    print("="*60)
    print("MODEL ARTIFACTS QUALITY CHECK")
    print("="*60)
    
    # ------------------------------
    # Check pipeline
    # ------------------------------
    
    print("\n1. Checking pipeline.pkl...")
    pipeline_path = Path(pipeline_path)
    print(f"   Looking in: {pipeline_path.resolve()}")
    
    if not pipeline_path.exists():
        print(f"Pipeline file not found: {pipeline_path}")
        print(f"   Current working directory: {Path.cwd()}")
        return False
    
    try:
        pipeline = joblib.load(pipeline_path)
        print(f"Pipeline loaded successfully")
    except Exception as e:
        print(f"Failed to load pipeline: {e}")
        return False
    
    # Check required steps
    required_steps = {"scaler", "model"}
    pipeline_steps = set(dict(pipeline.named_steps).keys())
    if not required_steps.issubset(pipeline_steps):
        print(f"Pipeline missing steps. Found: {pipeline_steps}")
        return False
    print(f"Pipeline contains required steps: {pipeline_steps}")
    
    # ------------------------------
    # Check meta
    # ------------------------------
    print("\n2. Checking meta.pkl...")
    meta_path = Path(meta_path)
    print(f"   Looking in: {meta_path.resolve()}")
    
    if not meta_path.exists():
        print(f"Meta file not found: {meta_path}")
        return False
    
    try:
        meta = joblib.load(meta_path)
        print(f"Meta loaded successfully")
    except Exception as e:
        print(f"Failed to load meta: {e}")
        return False
    
    # Expected keys in meta
    expected_keys = [
        "best_model_name", "selected_metric", "best_cv_score",
        "test_scores", "threshold", "feature_columns", "feature_importance", "per_model_best"
    ]
    missing_keys = [k for k in expected_keys if k not in meta]
    if missing_keys:
        print(f"Missing keys in meta: {missing_keys}")
        return False
    print(f"Meta contains all expected keys")
    
    # ------------------------------
    # Check feature importance
    # ------------------------------
    print("\n3. Checking feature importance / coefficients...")
    feature_importance = meta.get("feature_importance")
    if feature_importance is None:
        print("Feature importance is None")
    elif not feature_importance:
        print("Feature importance is empty")
    else:
        n_features_meta = len(feature_importance)
        n_features_pipeline = len(meta.get("feature_columns", []))
        if n_features_meta != n_features_pipeline:
            print(f"Mismatch in feature importance length: {n_features_meta} vs {n_features_pipeline}")
        else:
            print(f"Feature importance/coefficients look consistent ({n_features_meta} features)")
    
    # ------------------------------
    # Print model summary
    # ------------------------------
    print("\n4. Selected model and hyperparameters:")
    best_model_name = meta["best_model_name"]
    best_cv_score = meta["best_cv_score"]
    best_params = meta["per_model_best"][best_model_name]["best_params"]
    
    print(f"   Best model: {best_model_name}")
    print(f"   CV score: {best_cv_score:.4f}")
    print(f"   Hyperparameters: {best_params}")
    
    print("\n" + "="*60)
    print("MODEL ARTIFACTS QUALITY CHECK PASSED")
    print("="*60)

# Run the check
quality_check_artifacts()

MODEL ARTIFACTS QUALITY CHECK

1. Checking pipeline.pkl...
   Looking in: C:\Users\lcolx\Documents\MASTER\SCIENTIFIC_PROGRAMMING\FINAL_PROJECT\diabetes-scientific-programming\models\pipeline.pkl
Pipeline loaded successfully
Pipeline contains required steps: {'model', 'scaler'}

2. Checking meta.pkl...
   Looking in: C:\Users\lcolx\Documents\MASTER\SCIENTIFIC_PROGRAMMING\FINAL_PROJECT\diabetes-scientific-programming\models\meta.pkl
Meta loaded successfully
Meta contains all expected keys

3. Checking feature importance / coefficients...
Feature importance/coefficients look consistent (8 features)

4. Selected model and hyperparameters:
   Best model: random_forest
   CV score: 0.8456
   Hyperparameters: {'model__class_weight': 'balanced', 'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__n_estimators': 400}

MODEL ARTIFACTS QUALITY CHECK PASSED


## API checks

Verify if the Flask API is currently running.


In [65]:
def check_api_running(base_url="http://127.0.0.1:8000"):
    """Check if the API server is running by sending a test prediction and measuring latency."""
    print("=" * 60)
    print("API SERVER STATUS CHECK")
    print("=" * 60)
    
    test_data = {
        "Pregnancies": 2,
        "Glucose": 120,
        "BloodPressure": 70,
        "SkinThickness": 25,
        "Insulin": 100,
        "BMI": 28.5,
        "DiabetesPedigreeFunction": 0.5,
        "Age": 35
    }
    
    try:
        start_time = time.perf_counter()  # start timing
        response = requests.post(
            f"{base_url}/predict",
            json=test_data,
            headers={"Content-Type": "application/json"},
            timeout=5
        )
        end_time = time.perf_counter()  # end timing
        latency_ms = (end_time - start_time) * 1000  # milliseconds
        
        if response.status_code == 200:
            result = response.json()
            print(f"API is running at {base_url}")
            print(f"Sample prediction response: {result}")
            print(f"Latency: {latency_ms:.2f} ms")
            return True
        else:
            print(f"API responded with status code: {response.status_code}")
            print(f"Response text: {response.text}")
            print(f"Latency: {latency_ms:.2f} ms")
            return False
        
    except requests.exceptions.ConnectionError:
        print(f"API is NOT running at {base_url}")
        print(f"\nTo start the API, run:")
        print(f"   uvicorn api.app:app --reload")
        return False
    except requests.exceptions.Timeout:
        print(f"API request timed out")
        return False
    except Exception as e:
        print(f"Error connecting to API: {e}")
        return False

# Check API status
api_running = check_api_running()


API SERVER STATUS CHECK
✗ API request timed out


In [48]:
def test_api_error_handling(base_url="http://127.0.0.1:8000"):
    """Test API response to invalid input."""
    print("=" * 60)
    print("API ERROR HANDLING CHECK")
    print("=" * 60)
    
    invalid_data = {
        "Pregnancies": "invalid",  # Should be numeric
        "Glucose": 120
    }
    
    try:
        response = requests.post(
            f"{base_url}/predict",
            json=invalid_data,
            headers={"Content-Type": "application/json"},
            timeout=5
        )
        if response.status_code in [400, 422, 500]:
            print(f"API correctly handles invalid input (status: {response.status_code})")
        else:
            print(f"Unexpected response to invalid input: {response.status_code}")
    except Exception as e:
        print(f"Error handling test inconclusive: {e}")

# Run the error handling test
test_api_error_handling()


API ERROR HANDLING CHECK
✓ API correctly handles invalid input (status: 422)
